In [ ]:
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.txt.zip to train.txt (1).zip


In [ ]:
!unzip -o /content/train.txt.zip -d /content/

# Verify that train.txt now exists
!ls -F /content/

Archive:  /content/train.txt.zip
  inflating: /content/PoemDataset.csv  
 PoemDataset.csv   sample_data/  'train.txt (1).zip'   train.txt.zip


In [ ]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a default padding token
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a default padding token
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from datasets import load_dataset

def prepare_text_dataset_for_lm(file_path, tokenizer, block_size=128):
    # IMPORTANT: Ensure 'file_path' points to a plain text file, not a zip archive.
    # You need to manually unzip your zip file, e.g., to '/content/train.txt'.

    # Load the raw text dataset using the 'datasets' library
    # The 'text' builder can load a single text file.
    raw_datasets = load_dataset("text", data_files=file_path, split="train")

    # Tokenize the dataset
    def tokenize_function(examples):
        # examples["text"] is a list of strings when batched=True.
        # Pass the list of strings directly to the tokenizer for batch processing.
        # truncation=True will truncate each individual sequence to max_length if it exceeds it.
        return tokenizer(examples["text"], truncation=True, max_length=block_size)

    tokenized_datasets = raw_datasets.map(
        tokenize_function,
        batched=True,
        remove_columns=raw_datasets.column_names, # Remove all original columns
        desc="Running tokenizer on dataset",
    )

    # Group texts into fixed-size chunks (blocks) for language modeling
    # This function is adapted from Hugging Face examples for CLM
    def group_texts(examples):
        # Concatenate all texts.
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the small remainder to ensure all blocks are of 'block_size'.
        total_length = (total_length // block_size) * block_size
        # Split by chunks of block_size.
        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        # For causal language modeling, labels are just a copy of input_ids.
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(
        group_texts,
        batched=True,
        desc=f"Grouping texts in chunks of {block_size}",
    )

    return lm_datasets

train_dataset = prepare_text_dataset_for_lm(
    file_path="/content/PoemDataset.csv", # REMINDER: Ensure you have unzipped train.txt.zip to this path
    tokenizer=tokenizer
)

Generating train split: 0 examples [00:00, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/10001 [00:00<?, ? examples/s]

Grouping texts in chunks of 128:   0%|          | 0/10001 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    warmup_steps=100,
    prediction_loss_only=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,5.077407
200,4.942874
300,4.696272
400,4.730872
500,4.764353
600,4.771570
700,4.714582
800,4.646563
900,4.618679
1000,4.731258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14121, training_loss=4.346320031866221, metrics={'train_runtime': 1931.9639, 'train_samples_per_second': 14.618, 'train_steps_per_second': 7.309, 'total_flos': 1844852391936000.0, 'train_loss': 4.346320031866221, 'epoch': 3.0})

In [ ]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-finetuned/tokenizer_config.json', './gpt2-finetuned/tokenizer.json')

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer
)

prompt = "Happy"

output = generator(
    prompt,
    max_length=100,
    num_return_sequences=1,
    temperature=0.7
)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Happy.The old man says, ""There are things I've always liked about you.""So I say, ""What? I never liked you.""""I don't know.""So he says, ""You're the worst."""I say, ""You're the worst."""And he says, ""There's nothing I could ever do."""So he says, ""You're the worst."""And I say, ""There's nothing I could ever do."""And he says, ""There's nothing I could ever have."""And I say, ""There's nothing I could ever do."""And they say, ""That's how it goes."""And I say, ""That's how it goes.It always seems to go.I can't stop thinking about you.""The old man says, ""I have to stop thinking about you."""And I say, ""I have to stop thinking about you."""And he says, ""That's the way."""And he says, ""It's the way."""And I say, ""How it goes."""And I say, ""How it goes."""And he says, ""I don't know.""""And he says, ""How it goes."""And he says
